In [2]:
!pip install requests tqdm sentence_transformers pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import requests

# 1. Usamos la nueva ruta Raw que descubriste para el cohorte 2025
docs_url = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2025/01-intro/documents.json'
docs_response = requests.get(docs_url)

# 2. Leemos los datos en formato JSON
documents_raw = docs_response.json()

# 3. Aplanamos la lista de documentos
documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

print(f"Total de documentos procesados: {len(documents)}")
# Imprimimos el primero para verificar
documents[0]

Total de documentos procesados: 948


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Descargamos y cargamos el modelo pre-entrenado
# (La primera vez que corras esto, puede tardar unos segundos/minutos porque descargará el modelo de internet)
model = SentenceTransformer("all-mpnet-base-v2")

# 2. Hagamos una prueba con una oración simple
texto_prueba = "This is a simple sentence to test the model."

# 3. La función .encode() es la que hace la magia de Texto -> Números
vector = model.encode(texto_prueba)

# 4. Veamos el resultado
print(f"El vector tiene un tamaño de: {len(vector)} dimensiones (números).")
print(f"Aquí tienes un vistazo a los primeros 5 números de ese vector:")
print(vector[:5])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

El vector tiene un tamaño de: 768 dimensiones (números).
Aquí tienes un vistazo a los primeros 5 números de ese vector:
[ 0.00834174 -0.00839213 -0.02982975  0.02331362 -0.03813531]
(768,)


In [5]:
import numpy as np

# 1. Extraemos solo el campo "text" de cada documento y lo metemos en una lista simple
textos_documentos = [doc['text'] for doc in documents]

print("Iniciando la vectorización de los documentos. Esto puede tomar unos segundos/minutos dependiendo de tu compu...")

# 2. Codificamos todos los textos de golpe. 
# La librería nos devolverá automáticamente una matriz matemática de NumPy.
X = model.encode(textos_documentos, show_progress_bar=True)

# 3. Verificamos la "forma" (shape) de nuestra matriz
print(f"¡Listo! La matriz X tiene la forma: {X.shape}")

Iniciando la vectorización de los documentos. Esto puede tomar unos segundos/minutos dependiendo de tu compu...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

¡Listo! La matriz X tiene la forma: (948, 768)


In [6]:
# 1. Definimos nuestra pregunta (query)
query = "I just discovered the course. Can I still join it?"

# 2. Convertimos la pregunta en un vector usando nuestro traductor
v = model.encode(query)

# 3. Calculamos la similitud multiplicando la matriz X por nuestro vector v
# (Esto hace el Producto Punto contra los 948 documentos a la vez)
scores = X.dot(v)

# 4. Encontramos la posición (índice) del documento con el puntaje más alto
# argsort() ordena de menor a mayor. Tomamos el último [-1] porque es el mayor.
mejor_indice = scores.argsort()[-1]

# 5. Imprimimos los resultados
print(f"El puntaje de similitud (score) más alto es: {scores[mejor_indice]:.4f}\n")
print("La respuesta sugerida por el sistema es:")
print(documents[mejor_indice]['text'])

El puntaje de similitud (score) más alto es: 0.5173

La respuesta sugerida por el sistema es:
Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.
In order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.


In [7]:
import pandas as pd

class VectorSearchEngine():
    def __init__(self, documents, embeddings):
        self.documents = documents
        # Guardamos la matriz gigante X
        self.embeddings = embeddings

    def search(self, v_query, num_results=5):
        # 1. Hacemos el Producto Punto de la pregunta contra todos los documentos
        scores = self.embeddings.dot(v_query)
        
        # 2. Ordenamos los resultados del score más alto al más bajo y sacamos el Top K
        # np.argsort normalmente ordena de menor a mayor. 
        # Al ponerle el signo negativo (-scores), lo forzamos a ordenar de mayor a menor.
        idx = np.argsort(-scores)[:num_results]
        
        # 3. Preparamos los resultados para mostrarlos bonito
        resultados = []
        for i in idx:
            doc = self.documents[i].copy()
            doc['score'] = scores[i] # Agregamos el puntaje para verlo
            resultados.append(doc)
            
        return resultados

# ¡Instanciamos nuestro motor!
# Le pasamos la lista 'documents' y la matriz 'X' que creamos en los pasos anteriores.
search_engine = VectorSearchEngine(documents=documents, embeddings=X)

print("Motor de búsqueda inicializado y listo para recibir preguntas.")

Motor de búsqueda inicializado y listo para recibir preguntas.


In [11]:
# 1. Definimos la pregunta
query = "I just discovered the course. Can I still join it?"

# 2. Vectorizamos la pregunta con nuestro traductor (modelo)
v_query = model.encode(query)

# 3. Le pedimos a nuestro motor que busque los 5 mejores resultados
resultados_top5 = search_engine.search(v_query, num_results=5)

# 4. Usamos Pandas para mostrarlo como una tabla bonita
df_resultados = pd.DataFrame(resultados_top5)
df_resultados[['score', "text", 'course', 'section', 'question']]

,score,text,course,section,question
0,0.517271,"Yes, you can. You won’t be able to submit some...",machine-learning-zoomcamp,General course-related questions,The course has already started. Can I still jo...
1,0.511824,The course is available in the self-paced mode...,machine-learning-zoomcamp,General course-related questions,When does the next iteration start?
2,0.478565,You can get invitation code by coursera and us...,mlops-zoomcamp,Module 1: Introduction,IBM Cloud an alternative for AWS
3,0.467681,"The course videos are pre-recorded, you can st...",machine-learning-zoomcamp,General course-related questions,Is it going to be live? When?
4,0.443951,"No, you can only get a certificate if you fini...",data-engineering-zoomcamp,General course-related questions,Certificate - Can I follow the course in a sel...


In [12]:
from elasticsearch import Elasticsearch

# 1. Nos conectamos al motor que corre en Docker
es_client = Elasticsearch('http://localhost:9200') 

# 2. Definimos la nueva estructura. ¡OJO AL CAMPO text_vector!
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "text_vector": {
                "type": "dense_vector", # Le avisa a Elasticsearch que este campo no guardará palabras, sino un vector matemático (una lista larga de números con decimales).
                "dims": 768,            # Le especifica que cada lista tendrá exactamente 768 números (que es el tamaño que genera nuestro modelo de Hugging Face). Elasticsearch necesita saber este tamaño exacto para reservar la memoria adecuada.
                "index": True,          # ¡Esta es la clave para la velocidad! Le ordena a Elasticsearch que cree el "mapa de vecindarios" (índice vectorial) para estos números. Gracias a esto, podrá buscar a la velocidad de Pinecone en lugar de hacer comparaciones una por una.
                "similarity": "cosine"  # Le indica a la base de datos qué regla matemática usar para medir la distancia entre tu pregunta y los documentos. Aquí estamos eligiendo usar la Similitud del Coseno (que mide el ángulo entre las flechas) en lugar del Producto Punto.
            },
        }
    }
}

index_name = "course-questions-vectors"

# 3. Borramos el índice si ya existía (por si corres la celda dos veces)
es_client.indices.delete(index=index_name, ignore_unavailable=True)

# 4. Creamos el índice
es_client.indices.create(index=index_name, body=index_settings)

print("¡Índice vectorial creado con éxito en Elasticsearch!")

¡Índice vectorial creado con éxito en Elasticsearch!


In [13]:
from tqdm.auto import tqdm

print("Iniciando la carga de documentos a Elasticsearch...")

# Recorremos la lista de documentos y usamos 'enumerate' para saber en qué número (i) vamos
for i, doc in tqdm(enumerate(documents), total=len(documents)):
    
    # 1. Copiamos el documento original para no modificar la lista base
    doc_completo = doc.copy()
    
    # 2. Le agregamos el "código de barras matemático" (el vector)
    # Tomamos la fila 'i' de la matriz X y la convertimos a lista normal
    doc_completo['text_vector'] = X[i].tolist() 
    
    # 3. Lo enviamos al índice que acabamos de crear en Elasticsearch
    es_client.index(index=index_name, document=doc_completo)

print("¡Carga terminada!")

Iniciando la carga de documentos a Elasticsearch...


  0%|          | 0/948 [00:00<?, ?it/s]

¡Carga terminada!


In [15]:
# 1. Volvemos a usar nuestra pregunta
query = "I just discovered the course. Can I still join it?"

# 2. Generamos el vector.
v_query = model.encode(query)

# SOLUCIÓN AQUÍ: Convertimos el arreglo de NumPy en una lista pura de Python
# para que la librería de Elasticsearch no choque con la versión de NumPy.
v_query_list = v_query.tolist()

# 3. Armamos la estructura de la consulta usando la LISTA
es_query = {
    "field": "text_vector",  
    "query_vector": v_query_list, # <--- Pasamos la lista pura
    "k": 5,                  
    "num_candidates": 10000  
}

# 4. Hacemos la petición a la base de datos
res = es_client.search(
    index=index_name, 
    knn=es_query, 
    source=["text", "section", "question", "course"] 
)

# 5. Procesamos y mostramos los resultados
resultados_es = []
for hit in res['hits']['hits']:
    doc = hit['_source']
    doc['score'] = hit['_score']
    resultados_es.append(doc)

import pandas as pd
df_es = pd.DataFrame(resultados_es)
df_es[['score', 'course', 'section', 'question']]

,score,course,section,question
0,0.758636,machine-learning-zoomcamp,General course-related questions,The course has already started. Can I still jo...
1,0.755912,machine-learning-zoomcamp,General course-related questions,When does the next iteration start?
2,0.739282,mlops-zoomcamp,Module 1: Introduction,IBM Cloud an alternative for AWS
3,0.733840,machine-learning-zoomcamp,General course-related questions,Is it going to be live? When?
4,0.721975,data-engineering-zoomcamp,General course-related questions,Certificate - Can I follow the course in a sel...


In [ ]:
# 1. Usamos la misma estructura k-NN, pero le agregamos un bloque de "filter"
knn_query = {
    "field": "text_vector",
    "query_vector": v_query_list, # Usamos la lista que ya reparamos
    "k": 5,
    "num_candidates": 10000,
    "filter": {
        "term": {
            "course": "data-engineering-zoomcamp" # ¡Aquí aplicamos el filtro exacto!
        }
    }
}

# 2. Hacemos la búsqueda
res = es_client.search(
    index=index_name, 
    knn=knn_query,
    source=["text", "section", "question", "course"]
)

# 3. Procesamos y mostramos los resultados
resultados_es_filtrados = []
for hit in res['hits']['hits']:
    doc = hit['_source']
    doc['score'] = hit['_score']
    resultados_es_filtrados.append(doc)

# 4. Mostramos la tabla final.
df_es_filtrado = pd.DataFrame(resultados_es_filtrados)
df_es_filtrado[['score', 'course', 'section', 'question']]

,score,course,section,question
0,0.721975,data-engineering-zoomcamp,General course-related questions,Certificate - Can I follow the course in a sel...
1,0.711622,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...
2,0.688388,data-engineering-zoomcamp,General course-related questions,Environment - The GCP and other cloud provider...
3,0.686165,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?
4,0.680457,data-engineering-zoomcamp,General course-related questions,Course - Can I follow the course after it fini...
